# 15 树模型 / 集成模型 (core.models.sklearn_models / xgboost / lightgbm)

覆盖 RandomForestRiskModel / ExtraTreesRiskModel / GradientBoostingRiskModel /
XGBoostRiskModel / LightGBMRiskModel / CatBoostRiskModel + ModelExplainer。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from hscredit import (germancredit, init_setting, seed_everything,
    BaseRiskModel, RandomForestRiskModel, ExtraTreesRiskModel, GradientBoostingRiskModel,
)
from hscredit.core.metrics import ks, auc
seed_everything(42); init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('准备完成')

## 1. RandomForestRiskModel

In [ ]:
rf = RandomForestRiskModel(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_tr, y_tr)
prob_rf = rf.predict_proba(X_te)
print(f'RF  KS:{ks(y_te,prob_rf):.4f}  AUC:{auc(y_te,prob_rf):.4f}')
print('特征重要性:')
print(rf.feature_importances_.head(5))

## 2. ExtraTreesRiskModel

In [ ]:
et = ExtraTreesRiskModel(n_estimators=100, max_depth=6, random_state=42)
et.fit(X_tr, y_tr)
prob_et = et.predict_proba(X_te)
print(f'ET  KS:{ks(y_te,prob_et):.4f}  AUC:{auc(y_te,prob_et):.4f}')

## 3. GradientBoostingRiskModel

In [ ]:
gbm = GradientBoostingRiskModel(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
gbm.fit(X_tr, y_tr)
prob_gbm = gbm.predict_proba(X_te)
print(f'GBM KS:{ks(y_te,prob_gbm):.4f}  AUC:{auc(y_te,prob_gbm):.4f}')

## 4. XGBoostRiskModel

In [ ]:
try:
    from hscredit import XGBoostRiskModel
    xgb_model = XGBoostRiskModel(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)])
    prob_xgb = xgb_model.predict_proba(X_te)
    print(f'XGB KS:{ks(y_te,prob_xgb):.4f}  AUC:{auc(y_te,prob_xgb):.4f}')
    print('特征重要性:')
    print(xgb_model.feature_importances_.head(5))
except ImportError:
    print('xgboost 未安装，跳过')

## 5. LightGBMRiskModel

In [ ]:
try:
    from hscredit import LightGBMRiskModel
    lgb_model = LightGBMRiskModel(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        num_leaves=31, subsample=0.8, random_state=42,
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)])
    prob_lgb = lgb_model.predict_proba(X_te)
    print(f'LGB KS:{ks(y_te,prob_lgb):.4f}  AUC:{auc(y_te,prob_lgb):.4f}')
except ImportError:
    print('lightgbm 未安装，跳过')

## 6. ModelReport — 风控模型自动评估报告

In [ ]:
from hscredit import ModelReport
report = ModelReport(
    model=rf,
    X_train=X_tr, y_train=y_tr,
    X_test=X_te,  y_test=y_te,
)
report.fit()
print(report.metrics_summary())
fig = report.plot_ks(); plt.show()
fig = report.plot_roc(); plt.show()
fig = report.plot_feature_importance(); plt.show()

## 7. ModelExplainer — SHAP可解释性

In [ ]:
try:
    from hscredit.core.models.interpretability import ModelExplainer
    explainer = ModelExplainer(rf, feature_names=num_feats)
    shap_vals = explainer.compute_shap_values(X_te)
    print('SHAP值形状:', shap_vals.shape)
    fig = explainer.plot_shap_summary(X_te); plt.show()
    fig = explainer.plot_combined_importance(X_te, top_n=10); plt.show()
except Exception as e:
    print(f'SHAP分析: {e}')

## 8. 多模型指标汇总

In [ ]:
rows = []
for name, prob in [('RF', prob_rf), ('ET', prob_et), ('GBM', prob_gbm)]:
    rows.append({'模型': name, 'KS': round(ks(y_te,prob),4), 'AUC': round(auc(y_te,prob),4)})
pd.DataFrame(rows).sort_values('KS', ascending=False)